# Module 3: Deploying the ML Pipeline Debugger

Test locally → Build Docker image → Push to HuggingFace Spaces.

**Time:** ~20 min · **Difficulty:** Intermediate · **GPU:** Not required

> **Note:** HF Spaces deployment (Step 5) requires a HuggingFace account and `openenv push`.

In [ ]:
!pip install -q openenv-core torch numpy
import sys, os
sys.path.insert(0, os.path.abspath('ml-pipeline-debugger'))
print('Setup complete!')

## 1. Validate the Environment

Before deploying, run `openenv validate` to check the project is spec-compliant.

In [ ]:
import subprocess
result = subprocess.run(
    ['openenv', 'validate', '.'],
    cwd='ml-pipeline-debugger',
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('ERRORS:', result.stderr)

## 2. Test Locally with Uvicorn

The fastest way to verify the environment works before deploying.

In [ ]:
import subprocess, sys, time, os, urllib.request

env_vars = os.environ.copy()
env_vars['PYTHONPATH'] = os.path.abspath('ml-pipeline-debugger')

server = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'server.app:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd='ml-pipeline-debugger', env=env_vars,
    stdout=subprocess.PIPE, stderr=subprocess.PIPE,
)
time.sleep(4)

with urllib.request.urlopen('http://localhost:8000/health', timeout=5) as r:
    print('Health check:', r.read().decode())

# Verify all endpoints
for endpoint in ['/health', '/docs']:
    try:
        with urllib.request.urlopen(f'http://localhost:8000{endpoint}', timeout=5) as r:
            print(f'  {endpoint}: {r.status} OK')
    except Exception as e:
        print(f'  {endpoint}: {e}')

In [ ]:
# Quick end-to-end smoke test
from client import MLPipelineDebuggerEnv
from models import MLDebugAction

with MLPipelineDebuggerEnv(base_url='http://localhost:8000').sync() as env:
    obs = env.reset().observation
    print(f'reset() OK  — task={obs.task_id}, difficulty={obs.difficulty}')
    result = env.step(MLDebugAction(
        fixed_code=obs.broken_script,
        explanation='smoke test',
        task_id=obs.task_id
    ))
    print(f'step() OK   — reward={result.reward}, done={result.done}')
    state = env.state()
    print(f'state() OK  — step_count={state.step_count}')

print('\nAll endpoints working. Ready to deploy!')

In [ ]:
server.terminate(); server.wait(); print('Local server stopped.')

## 3. Build and Run with Docker

Docker gives full isolation — identical to what HF Spaces runs.

In [ ]:
# Build the Docker image
result = subprocess.run(
    ['docker', 'build', '-t', 'ml-pipeline-debugger:latest', '-f', 'server/Dockerfile', '.'],
    cwd='ml-pipeline-debugger',
    capture_output=True, text=True
)
print('Build stdout:', result.stdout[-500:] if result.stdout else '')
print('Build return code:', result.returncode)

In [ ]:
# Run the container
container = subprocess.Popen(
    ['docker', 'run', '--rm', '-p', '8001:8000',
     '-e', 'WORKERS=2',
     '-e', 'MAX_CONCURRENT_ENVS=10',
     'ml-pipeline-debugger:latest'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE,
)
time.sleep(8)  # Docker takes longer to start

with urllib.request.urlopen('http://localhost:8001/health', timeout=5) as r:
    print('Docker container health:', r.read().decode())

In [ ]:
# Test the containerised environment
with MLPipelineDebuggerEnv(base_url='http://localhost:8001').sync() as env:
    obs = env.reset().observation
    print(f'Docker environment OK — task={obs.task_id}, bug={obs.bug_type}')

container.terminate(); container.wait()
print('Container stopped.')

## 4. Deploy to HuggingFace Spaces

One command pushes to HF Spaces and makes the environment publicly available.

In [ ]:
# Set your HuggingFace username
HF_USERNAME = 'your-username'  # <-- change this

print(f'Command to deploy:')
print(f'  cd ml-pipeline-debugger')
print(f'  openenv push --repo-id {HF_USERNAME}/ml-pipeline-debugger')
print()
print('After deployment your environment will be live at:')
print(f'  https://{HF_USERNAME}-ml-pipeline-debugger.hf.space')

# Uncomment to deploy:
# result = subprocess.run(
#     ['openenv', 'push', '--repo-id', f'{HF_USERNAME}/ml-pipeline-debugger'],
#     cwd='ml-pipeline-debugger'
# )

## 5. Connect to the Deployed Space

In [ ]:
HF_SPACE_URL = f'https://{HF_USERNAME}-ml-pipeline-debugger.hf.space'

# Uncomment after deploying:
# with MLPipelineDebuggerEnv(base_url=HF_SPACE_URL).sync() as env:
#     obs = env.reset().observation
#     print(f'Live Space: task={obs.task_id}, difficulty={obs.difficulty}')
#     print(f'Symptom: {obs.task_description[:100]}')

print('Deployment complete!')
print('Submit your Space URL to the hackathon dashboard.')

## Summary

| Method | Command | URL |
|--------|---------|-----|
| Uvicorn | `uvicorn server.app:app --port 8000` | `http://localhost:8000` |
| Docker | `docker run -p 8000:8000 ml-pipeline-debugger:latest` | `http://localhost:8000` |
| HF Spaces | `openenv push --repo-id user/ml-pipeline-debugger` | `https://user-ml-pipeline-debugger.hf.space` |

**Next:** Module 4 — How the environment was built from scratch (the selected PS).